In [2]:
!pip install torchvision

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 29.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 12.7 MB/s  0:00:43m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 27.0 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 23.1 MB/s  0:00:22m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 9.0 MB/s  0:00:016m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 6.5 MB/s  0:00:136m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 5.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 23.4 MB/s  0:00:26m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 17.2 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 10.6 MB/s  0:00:

In [4]:
import os
from PIL import Image
import numpy as np
from torchvision import datasets, transforms
import random

# ----------
# CONFIG
# ----------
OUTPUT_DIR = "favicons_rr_candidates"
NUM_FAVICONS = 50          # how many candidate favicons to generate
FAVICON_SIZE = 32          # 32x32 is typical
BACKGROUND_COLOR = 255     # white background
FOREGROUND_COLOR = 0       # black foreground
TARGET_CHAR = 'R'          # visually we want two "R"s (EMNIST uses lowercase label)
# Minimum properties for a "good" glyph
MIN_ON_PIXELS = 30         # minimum number of dark pixels to consider it a real letter
MIN_VERTICAL_SPAN = 10     # minimum vertical span (in pixels) of ink
# ----------

def center_and_resize(img_array, target_size):
    """
    Take a 28x28 EMNIST image, center it on a square canvas,
    resize to target_size (for each glyph), and return a PIL.Image.
    """
    # Convert from float(0–1) to uint8(0–255) if needed
    if img_array.max() <= 1.0:
        img_array = img_array * 255.0
    img_array = img_array.astype(np.uint8)

    # EMNIST is white-on-black, invert to black-on-white
    img_array = 255 - img_array

    img = Image.fromarray(img_array, mode='L')

    padded_size = 32
    canvas = Image.new("L", (padded_size, padded_size), color=BACKGROUND_COLOR)
    x_off = (padded_size - img.width) // 2
    y_off = (padded_size - img.height) // 2
    canvas.paste(img, (x_off, y_off))

    glyph = canvas.resize((target_size, target_size), Image.BOX)
    return glyph

def is_reasonable_glyph(img_array):
    """
    Heuristic filter: keep only images that look like a real letter.
    """
    # img_array is 28x28 float or uint8 (but we don't care about absolute values)
    if img_array.max() <= 1.0:
        arr = (img_array * 255.0).astype(np.uint8)
    else:
        arr = img_array.astype(np.uint8)

    # Invert to have black ink on white background
    arr = 255 - arr

    # Count dark pixels
    dark = arr < 200  # threshold for "ink"
    num_dark = dark.sum()
    if num_dark < MIN_ON_PIXELS:
        return False

    # Vertical span of ink
    ys, xs = np.where(dark)
    if len(ys) == 0:
        return False
    vertical_span = ys.max() - ys.min()
    if vertical_span < MIN_VERTICAL_SPAN:
        return False

    return True

def collect_r_images(dataset, max_samples=1000):
    """
    Collect many 'r' images that pass our heuristic filter.
    """
    # EMNIST letters labels: 1–26 -> 'a'–'z'
    # So 'r' index = ord('r') - ord('a') + 1
    target_label = ord('r') - ord('a') + 1

    r_images = []
    for img_t, label in dataset:
        if label == target_label:
            arr = img_t.squeeze().numpy()
            if is_reasonable_glyph(arr):
                r_images.append(arr)
                if len(r_images) >= max_samples:
                    break

    return r_images

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    transform = transforms.ToTensor()
    dataset = datasets.EMNIST(
        root="./data",
        split="letters",
        train=True,
        download=True,
        transform=transform
    )

    # Collect many candidate 'r' images
    r_images = collect_r_images(dataset, max_samples=2000)
    print(f"Collected {len(r_images)} candidate 'r' images that look reasonable.")

    if len(r_images) < 2:
        raise RuntimeError("Not enough 'r' samples collected from EMNIST to build favicons.")

    glyph_size = FAVICON_SIZE // 2

    # Generate multiple favicons with random pairs of 'r's
    for i in range(NUM_FAVICONS):
        img1, img2 = random.sample(r_images, 2)

        glyph1 = center_and_resize(img1, glyph_size)
        glyph2 = center_and_resize(img2, glyph_size)

        favicon = Image.new("L", (FAVICON_SIZE, FAVICON_SIZE), color=BACKGROUND_COLOR)
        favicon.paste(glyph1, (0, 0))
        favicon.paste(glyph2, (glyph_size, 0))

        png_path = os.path.join(OUTPUT_DIR, f"favicon_rr_{i:03d}.png")
        ico_path = os.path.join(OUTPUT_DIR, f"favicon_rr_{i:03d}.ico")

        favicon.save(png_path)
        favicon.save(ico_path, format="ICO", sizes=[(FAVICON_SIZE, FAVICON_SIZE)])

        print(f"Saved {png_path} and {ico_path}")

    print("Done. Inspect the PNGs in the output directory and pick your favorite.")

main()


Collected 2000 candidate 'r' images that look reasonable.
Saved favicons_rr_candidates/favicon_rr_000.png and favicons_rr_candidates/favicon_rr_000.ico
Saved favicons_rr_candidates/favicon_rr_001.png and favicons_rr_candidates/favicon_rr_001.ico
Saved favicons_rr_candidates/favicon_rr_002.png and favicons_rr_candidates/favicon_rr_002.ico
Saved favicons_rr_candidates/favicon_rr_003.png and favicons_rr_candidates/favicon_rr_003.ico
Saved favicons_rr_candidates/favicon_rr_004.png and favicons_rr_candidates/favicon_rr_004.ico
Saved favicons_rr_candidates/favicon_rr_005.png and favicons_rr_candidates/favicon_rr_005.ico
Saved favicons_rr_candidates/favicon_rr_006.png and favicons_rr_candidates/favicon_rr_006.ico
Saved favicons_rr_candidates/favicon_rr_007.png and favicons_rr_candidates/favicon_rr_007.ico
Saved favicons_rr_candidates/favicon_rr_008.png and favicons_rr_candidates/favicon_rr_008.ico
Saved favicons_rr_candidates/favicon_rr_009.png and favicons_rr_candidates/favicon_rr_009.ico
Sa

In [15]:
from PIL import Image, ImageOps
import os

INPUT_FILE = "favicons_rr_candidates/favicon_rr_024.png"
OUTPUT_DIR = "final"
OUTPUT_32 = os.path.join(OUTPUT_DIR, "favicon-32x32.png")
OUTPUT_16 = os.path.join(OUTPUT_DIR, "favicon-16x16.png")

def make_png_favicons(input_path, out_32, out_16):
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Input file not found: {input_path}")

    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(out_32), exist_ok=True)

    # Open as grayscale
    img = Image.open(input_path).convert("L")

    # Invert: black ↔ white
    img_inv = ImageOps.invert(img)

    # Make 32x32 and 16x16
    img_32 = img_inv.resize((32, 32), Image.LANCZOS)
    img_16 = img_inv.resize((16, 16), Image.LANCZOS)

    # Save PNGs
    img_32.save(out_32)
    img_16.save(out_16)

    print(f"Saved {out_32}")
    print(f"Saved {out_16}")

if __name__ == "__main__":
    make_png_favicons(INPUT_FILE, OUTPUT_32, OUTPUT_16)



Saved final/favicon-32x32.png
Saved final/favicon-16x16.png


In [17]:
from PIL import Image, ImageOps
import os

INPUT_FILE = "favicons_rr_candidates/favicon_rr_024.png"
OUTPUT_DIR = "final"
OUTPUT_32 = os.path.join(OUTPUT_DIR, "favicon-32x32.png")
OUTPUT_16 = os.path.join(OUTPUT_DIR, "favicon-16x16.png")

# how many pixels to push the content down *before* resizing
VERTICAL_SHIFT = 8  # tweak this (2–6) until it looks right

def shift_down(img, pixels, bg=0):
    """
    Shift image content down by `pixels`, filling new space with `bg`.
    Assumes grayscale ('L').
    """
    w, h = img.size
    shifted = Image.new("L", (w, h), color=bg)
    # paste original image starting `pixels` down
    shifted.paste(img, (0, pixels))
    return shifted

def make_png_favicons(input_path, out_32, out_16):
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Input file not found: {input_path}")

    os.makedirs(os.path.dirname(out_32), exist_ok=True)

    # Open as grayscale
    img = Image.open(input_path).convert("L")

    # Invert: black ↔ white (so R's are white on black)
    img_inv = ImageOps.invert(img)

    # Shift content downward BEFORE resizing
    # Background for the shifted image should be black (0) since our bg is black
    img_shifted = shift_down(img_inv, VERTICAL_SHIFT, bg=0)

    # Make 32x32 and 16x16 from shifted image
    img_32 = img_shifted.resize((32, 32), Image.LANCZOS)
    img_16 = img_shifted.resize((16, 16), Image.LANCZOS)

    img_32.save(out_32)
    img_16.save(out_16)

    print(f"Saved {out_32}")
    print(f"Saved {out_16}")

if __name__ == "__main__":
    make_png_favicons(INPUT_FILE, OUTPUT_32, OUTPUT_16)


Saved final/favicon-32x32.png
Saved final/favicon-16x16.png
